# AI-Powered Customer Feedback Intelligence

End-to-end Microsoft Fabric pipeline: Bronze → Silver → Gold, rule-based baseline, DistilBERT sentiment, final Gold table for Power BI.

Run in a Fabric notebook attached to the `CustomerFeedbackLH` lakehouse.

## 01 · Bronze: ingest synthetic customer reviews into a Delta table

In [ ]:

from pyspark.sql import functions as F

# 1. Create synthetic customer feedback
reviews = [
    (1, "The delivery was fast and the product is excellent", "Electronics", 5, "2026-10-01", "Web"),
    (2, "Battery life is terrible and does not last", "Electronics", 1, "2026-10-01", "Mobile"),
    (3, "Good quality for the price", "Home", 4, "2026-10-02", "Web"),
    (4, "Customer support never responded to my complaint", "Services", 1, "2026-10-02", "Mobile"),
    (5, "Very happy with the purchase", "Fashion", 5, "2026-10-03", "Web"),
    (6, "The package arrived damaged", "Home", 2, "2026-10-03", "Partner"),
    (7, "Easy to use and works as expected", "Electronics", 4, "2026-10-04", "Mobile"),
    (8, "Too expensive for this quality", "Fashion", 2, "2026-10-04", "Web"),
    (9, "Excellent customer service", "Services", 5, "2026-10-05", "Partner"),
    (10, "The app keeps crashing", "Electronics", 1, "2026-10-05", "Mobile"),
    (11, "Delivery was late but the product is fine", "Home", 3, "2026-10-06", "Web"),
    (12, "Great value and quick delivery", "Fashion", 5, "2026-10-06", "Partner"),
    (13, "Instructions were confusing", "Home", 2, "2026-10-07", "Mobile"),
    (14, "The product exceeded my expectations", "Electronics", 5, "2026-10-07", "Web"),
    (15, "Refund process was very slow", "Services", 2, "2026-10-08", "Partner"),
    (16, "Comfortable and well made", "Fashion", 4, "2026-10-08", "Mobile"),
    (17, "Not working after two days", "Electronics", 1, "2026-10-09", "Web"),
    (18, "Helpful support team resolved my issue", "Services", 4, "2026-10-09", "Mobile"),
    (19, "Average product, nothing special", "Home", 3, "2026-10-10", "Partner"),
    (20, "Fantastic experience from start to finish", "Fashion", 5, "2026-10-10", "Web")
]

columns = [
    "record_id",
    "review_text",
    "product_category",
    "rating",
    "review_date",
    "source_system"
]

df = spark.createDataFrame(reviews, columns)

# 2. Add ingestion metadata
df = (
    df.withColumn("review_date", F.to_date("review_date", "yyyy-MM-dd"))
      .withColumn("ingested_at", F.current_timestamp())
)

# 3. Inspect the incoming dataset
print(f"Incoming record count: {df.count()}")
display(df)


In [ ]:

# Write the initial dataset to the attached Lakehouse
(
    df.write
      .format("delta")
      .mode("overwrite")
      .saveAsTable("bronze_customer_feedback")
)

print("Bronze ingestion completed.")

spark.sql("""
    SELECT COUNT(*) AS total_rows
    FROM bronze_customer_feedback
""").show()

spark.sql("""
    SELECT *
    FROM bronze_customer_feedback
    ORDER BY record_id
    LIMIT 5
""").show(truncate=False)


## 02 · Silver: data-quality rules, deduplication and quarantine

In [ ]:

from pyspark.sql import functions as F
from functools import reduce

# 1. Read the persisted Bronze table
bronze_df = spark.table("bronze_customer_feedback")

# 2. Create test records to demonstrate quality handling
# ID 2 is a duplicate; IDs 21-23 contain invalid data.
test_records = [
    (2, "Duplicate review record", "Electronics", 4, "2026-10-01", "Web"),
    (21, "Rating outside valid range", "Home", 6, "2026-10-11", "Web"),
    (22, None, "Fashion", 4, "2026-10-11", "Mobile"),
    (23, "Invalid date review", "Services", 3, "not-a-date", "Partner")
]

test_df = spark.createDataFrame(
    test_records,
    [
        "record_id", "review_text", "product_category",
        "rating", "review_date", "source_system"
    ]
).withColumn(
    "review_date", F.to_date("review_date", "yyyy-MM-dd")
).withColumn(
    "ingested_at", F.current_timestamp()
)

# 3. Combine Bronze records with test records
combined_df = bronze_df.unionByName(test_df)

print("Bronze records:", bronze_df.count())
print("Records including test cases:", combined_df.count())


In [ ]:

# 1. Define quality rules
checked_df = (
    combined_df
    .withColumn(
        "quality_errors",
        F.concat_ws(
            "; ",
            F.when(
                F.col("record_id").isNull(),
                F.lit("Missing record_id")
            ),
            F.when(
                F.col("review_text").isNull() |
                (F.trim(F.col("review_text")) == ""),
                F.lit("Missing review_text")
            ),
            F.when(
                F.col("product_category").isNull() |
                (F.trim(F.col("product_category")) == ""),
                F.lit("Missing product_category")
            ),
            F.when(
                F.col("source_system").isNull() |
                (F.trim(F.col("source_system")) == ""),
                F.lit("Missing source_system")
            ),
            F.when(
                F.col("rating").isNull() |
                (~F.col("rating").between(1, 5)),
                F.lit("Rating must be between 1 and 5")
            ),
            F.when(
                F.col("review_date").isNull(),
                F.lit("Missing or invalid review_date")
            )
        )
    )
)

# 2. Separate records with basic validation failures
invalid_df = checked_df.filter(F.col("quality_errors") != "")
valid_candidates = checked_df.filter(F.col("quality_errors") == "")

# 3. Identify duplicate IDs among otherwise valid records
from pyspark.sql.window import Window

dedup_window = Window.partitionBy("record_id").orderBy(
    F.col("ingested_at").asc()
)

ranked_df = valid_candidates.withColumn(
    "duplicate_rank", F.row_number().over(dedup_window)
)

duplicate_df = ranked_df.filter(F.col("duplicate_rank") > 1)
silver_df = ranked_df.filter(F.col("duplicate_rank") == 1)

# 4. Add duplicate records to quarantine with a rejection reason
duplicate_quarantine = (
    duplicate_df
    .withColumn("quality_errors", F.lit("Duplicate record_id"))
    .drop("duplicate_rank")
)

invalid_quarantine = invalid_df

# 5. Create final quarantine dataset
quarantine_df = invalid_quarantine.unionByName(
    duplicate_quarantine
)

# 6. Remove internal processing columns from Silver
silver_df = silver_df.drop(
    "quality_errors", "duplicate_rank"
)

print("Valid unique records:", silver_df.count())
print("Quarantined records:", quarantine_df.count())


In [ ]:

# Persist cleaned records
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_customer_feedback")
)

# Persist rejected records with their quality errors
(
    quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("quarantine_customer_feedback")
)

print("Silver and quarantine tables saved.")


In [ ]:

print("=== DATA QUALITY SUMMARY ===")

spark.sql("""
    SELECT 'Bronze' AS layer, COUNT(*) AS record_count
    FROM bronze_customer_feedback

    UNION ALL

    SELECT 'Silver', COUNT(*)
    FROM silver_customer_feedback

    UNION ALL

    SELECT 'Quarantine', COUNT(*)
    FROM quarantine_customer_feedback
""").show()

print("=== QUARANTINE REASONS ===")

spark.sql("""
    SELECT record_id, quality_errors
    FROM quarantine_customer_feedback
    ORDER BY record_id
""").show(truncate=False)


## 03 · Gold: business KPI tables (overview, category, daily, source)

In [ ]:

from pyspark.sql import functions as F

# Read clean Silver data
silver_df = spark.table("silver_customer_feedback")

# Business rule: ratings 1–2 are low; ratings 4–5 are high
low_rating = F.col("rating") <= 2
high_rating = F.col("rating") >= 4

# 1. Overall business KPIs
gold_overview = silver_df.agg(
    F.count("*").alias("total_reviews"),
    F.round(F.avg("rating"), 2).alias("average_rating"),
    F.sum(F.when(low_rating, 1).otherwise(0)).alias("low_rating_reviews"),
    F.sum(F.when(high_rating, 1).otherwise(0)).alias("high_rating_reviews")
).withColumn(
    "low_rating_pct",
    F.round(
        F.col("low_rating_reviews") / F.col("total_reviews") * 100, 2
    )
).withColumn(
    "high_rating_pct",
    F.round(
        F.col("high_rating_reviews") / F.col("total_reviews") * 100, 2
    )
)

# 2. Performance by product category
gold_category = silver_df.groupBy("product_category").agg(
    F.count("*").alias("total_reviews"),
    F.round(F.avg("rating"), 2).alias("average_rating"),
    F.sum(F.when(low_rating, 1).otherwise(0)).alias("low_rating_reviews"),
    F.sum(F.when(high_rating, 1).otherwise(0)).alias("high_rating_reviews")
).withColumn(
    "low_rating_pct",
    F.round(F.col("low_rating_reviews") / F.col("total_reviews") * 100, 2)
).orderBy("average_rating")

# 3. Daily customer feedback trends
gold_daily = silver_df.groupBy("review_date").agg(
    F.count("*").alias("total_reviews"),
    F.round(F.avg("rating"), 2).alias("average_rating"),
    F.sum(F.when(low_rating, 1).otherwise(0)).alias("low_rating_reviews")
).orderBy("review_date")

# 4. Feedback by source system
gold_source = silver_df.groupBy("source_system").agg(
    F.count("*").alias("total_reviews"),
    F.round(F.avg("rating"), 2).alias("average_rating"),
    F.sum(F.when(low_rating, 1).otherwise(0)).alias("low_rating_reviews")
).orderBy("average_rating")

# Save Gold tables as Delta tables
gold_overview.write.format("delta").mode("overwrite").saveAsTable(
    "gold_feedback_overview"
)
gold_category.write.format("delta").mode("overwrite").saveAsTable(
    "gold_category_metrics"
)
gold_daily.write.format("delta").mode("overwrite").saveAsTable(
    "gold_daily_trends"
)
gold_source.write.format("delta").mode("overwrite").saveAsTable(
    "gold_source_metrics"
)

print("Gold business metrics created successfully!")


In [ ]:

print("=== OVERALL CUSTOMER EXPERIENCE ===")
spark.table("gold_feedback_overview").show(truncate=False)

print("=== PRODUCT CATEGORY PERFORMANCE ===")
spark.table("gold_category_metrics").show(truncate=False)

print("=== DAILY FEEDBACK TRENDS ===")
spark.table("gold_daily_trends").show(truncate=False)

print("=== FEEDBACK BY SOURCE ===")
spark.table("gold_source_metrics").show(truncate=False)


## 04 · Rule-based baseline: keyword sentiment and themes (v1 and improved v2)

In [ ]:

from pyspark.sql import functions as F

silver_df = spark.table("silver_customer_feedback")

text = F.lower(F.coalesce(F.col("review_text"), F.lit("")))

# Baseline sentiment classification
enriched_df = silver_df.withColumn(
    "sentiment",
    F.when(
        text.rlike(
            r"\b(terrible|never responded|damaged|crashing|"
            r"too expensive|slow|not working|confusing)\b"
        ),
        "Negative"
    ).when(
        text.rlike(
            r"\b(excellent|happy|great|fantastic|helpful|"
            r"love|comfortable|exceeded|quick|fast)\b"
        ),
        "Positive"
    ).otherwise("Neutral")
)

# Baseline complaint theme classification
enriched_df = enriched_df.withColumn(
    "feedback_theme",
    F.when(
        text.rlike(r"\b(delivery|delivered|arrived|package)\b"),
        "Delivery"
    ).when(
        text.rlike(r"\b(battery|product|quality|working|instructions)\b"),
        "Product Quality"
    ).when(
        text.rlike(r"\b(expensive|price|value)\b"),
        "Pricing"
    ).when(
        text.rlike(r"\b(support|service|complaint|refund|team)\b"),
        "Customer Service"
    ).when(
        text.rlike(r"\b(app|crashing|technical|software)\b"),
        "App/Technical Issue"
    ).otherwise("Other")
).withColumn(
    "enrichment_method",
    F.lit("Keyword baseline")
).withColumn(
    "enrichment_status",
    F.lit("Completed")
)

# Persist enriched data as a Gold Delta table
(
    enriched_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_customer_feedback_enriched")
)

print("Enrichment baseline created successfully.")


In [ ]:

spark.table("gold_customer_feedback_enriched").select(
    "record_id",
    "review_text",
    "rating",
    "sentiment",
    "feedback_theme",
    "enrichment_method",
    "enrichment_status"
).orderBy("record_id").show(25, truncate=False)


In [ ]:
from pyspark.sql import functions as F

positive_pattern = (
    r"good quality|easy to use|works as expected|"
    r"excellent|great|love|amazing|fantastic|"
    r"happy|satisfied|exceeded my expectations|"
    r"good|comfortable|well made|helpful|"
    r"resolved my issue|great value|recommend|"
    r"fast delivery|quick delivery"
)

negative_pattern = (
    r"terrible|poor|broken|defective|"
    r"late delivery|arrived damaged|damaged|"
    r"disappointed|waste|does not work|"
    r"not working|never responded|"
    r"keeps crashing|too expensive|"
    r"confusing|very slow|slow|not worth"
)

silver_df = spark.table("silver_customer_feedback")

improved_df = silver_df.withColumn(
    "sentiment",
    F.when(
        F.lower(F.col("review_text")).rlike(negative_pattern),
        "Negative"
    ).when(
        F.lower(F.col("review_text")).rlike(positive_pattern),
        "Positive"
    ).otherwise("Neutral")
)

print("Total reviews:", improved_df.count())
improved_df.select(
    "record_id", "review_text", "sentiment"
).orderBy("record_id").show(20, truncate=False)


In [ ]:
(
    improved_df
    .withColumn("enrichment_method", F.lit("Improved keyword baseline"))
    .withColumn("enrichment_status", F.lit("Completed"))
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_customer_feedback_enriched_v2")
)

print("Improved baseline saved successfully.")

spark.table("gold_customer_feedback_enriched_v2") \
    .groupBy("sentiment") \
    .count() \
    .orderBy("sentiment") \
    .show()


## 05 · AI: DistilBERT sentiment inference with Hugging Face Transformers

In [ ]:

%pip install transformers torch


In [ ]:

import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

print("Hugging Face Xet downloads disabled for this session.")


In [ ]:

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Libraries loaded successfully.")


In [ ]:

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

tests = [
    "The delivery was fast and the product is excellent",
    "Battery life is terrible and does not last"
]

for review, result in zip(tests, sentiment_model(tests, truncation=True)):
    print("Review:", review)
    print("Sentiment:", result["label"])
    print("Confidence:", round(result["score"], 4))
    print("-" * 40)


In [ ]:

from transformers import pipeline
from pyspark.sql import functions as F

# Load the real sentiment model
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

# Read clean Silver records
silver_df = spark.table("silver_customer_feedback")

# This demo dataset is small (20 rows), so collect it for batch inference.
# For large production datasets, use distributed batch inference instead.
reviews = (
    silver_df
    .select("record_id", "review_text")
    .orderBy("record_id")
    .collect()
)

# Run actual model inference
predictions = sentiment_model(
    [row["review_text"] for row in reviews],
    truncation=True,
    batch_size=8
)

# Combine each prediction with its original record ID
results = [
    (
        row["record_id"],
        row["review_text"],
        prediction["label"].capitalize(),
        float(prediction["score"]),
        "distilbert-base-uncased-finetuned-sst-2-english"
    )
    for row, prediction in zip(reviews, predictions)
]

result_schema = [
    "record_id",
    "review_text",
    "ai_sentiment",
    "sentiment_confidence",
    "model_name"
]

ai_df = spark.createDataFrame(results, result_schema)

# Add processing timestamp and persist results
ai_df = ai_df.withColumn(
    "processed_at", F.current_timestamp()
)

(
    ai_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_ai_sentiment")
)

print("AI sentiment analysis completed.")
print("Reviews processed:", len(results))


In [ ]:

print("=== AI SENTIMENT RESULTS ===")

spark.table("gold_ai_sentiment").select(
    "record_id",
    "review_text",
    "ai_sentiment",
    "sentiment_confidence"
).orderBy("record_id").show(25, truncate=False)

print("=== SENTIMENT SUMMARY ===")

spark.table("gold_ai_sentiment").groupBy(
    "ai_sentiment"
).agg(
    F.count("*").alias("review_count"),
    F.round(F.avg("sentiment_confidence"), 4).alias(
        "average_model_confidence"
    )
).show()


## 06 · Compare baseline vs AI and build the final Gold table

In [ ]:

from pyspark.sql import functions as F

baseline_df = spark.table(
    "gold_customer_feedback_enriched"
).select(
    "record_id",
    F.col("sentiment").alias("baseline_sentiment")
)

ai_df = spark.table(
    "gold_ai_sentiment"
).select(
    "record_id",
    "ai_sentiment"
)

comparison_df = baseline_df.join(
    ai_df,
    on="record_id",
    how="inner"
).withColumn(
    "prediction_match",
    F.col("baseline_sentiment") == F.col("ai_sentiment")
)

print("=== BASELINE VS AI ===")
comparison_df.orderBy("record_id").show(truncate=False)

print("=== AGREEMENT SUMMARY ===")
comparison_df.agg(
    F.count("*").alias("total_reviews"),
    F.sum(
        F.when(F.col("prediction_match"), 1).otherwise(0)
    ).alias("matching_predictions"),
    F.round(
        F.avg(
            F.when(F.col("prediction_match"), 1.0).otherwise(0.0)
        ) * 100,
        2
    ).alias("agreement_percentage")
).show()


In [ ]:
from pyspark.sql import functions as F

baseline_df = spark.table(
    "gold_customer_feedback_enriched"
).select(
    "record_id",
    "review_text",
    F.col("sentiment").alias("baseline_sentiment")
)

ai_df = spark.table(
    "gold_ai_sentiment"
).select(
    "record_id",
    "ai_sentiment",
    "sentiment_confidence"
)

comparison_df = baseline_df.join(
    ai_df,
    on="record_id",
    how="inner"
)

print("=== THE 4 DISAGREEMENTS ===")

comparison_df.filter(
    F.col("baseline_sentiment") != F.col("ai_sentiment")
).select(
    "record_id",
    "review_text",
    "baseline_sentiment",
    "ai_sentiment",
    "sentiment_confidence"
).orderBy("record_id").show(
    truncate=False
)


In [ ]:
from pyspark.sql import functions as F

baseline_v2 = spark.table(
    "gold_customer_feedback_enriched_v2"
).select(
    "record_id",
    F.col("sentiment").alias("baseline_sentiment")
)

ai_df = spark.table(
    "gold_ai_sentiment"
).select(
    "record_id",
    "ai_sentiment"
)

comparison_v2 = baseline_v2.join(
    ai_df,
    on="record_id",
    how="inner"
).withColumn(
    "prediction_match",
    F.col("baseline_sentiment") == F.col("ai_sentiment")
)

print("=== IMPROVED BASELINE VS AI ===")

comparison_v2.agg(
    F.count("*").alias("total_reviews"),
    F.sum(
        F.when(F.col("prediction_match"), 1).otherwise(0)
    ).alias("matching_predictions"),
    F.round(
        F.avg(
            F.when(F.col("prediction_match"), 1.0).otherwise(0.0)
        ) * 100,
        2
    ).alias("agreement_percentage")
).show()

print("=== COMPARISON BY REVIEW ===")

comparison_v2.orderBy("record_id").show(truncate=False)


In [ ]:
from pyspark.sql import functions as F

# Improved baseline sentiment
baseline_df = spark.table(
    "gold_customer_feedback_enriched_v2"
).select(
    "record_id",
    "review_text",
    "product_category",
    "rating",
    "review_date",
    "source_system",
    F.col("sentiment").alias("baseline_sentiment")
)

# Original rule-based themes
themes_df = spark.table(
    "gold_customer_feedback_enriched"
).select(
    "record_id",
    "feedback_theme"
)

# AI predictions
ai_df = spark.table(
    "gold_ai_sentiment"
).select(
    "record_id",
    "ai_sentiment",
    "sentiment_confidence",
    "model_name",
    "processed_at"
)

# Combine the three datasets
final_df = (
    baseline_df
    .join(themes_df, on="record_id", how="inner")
    .join(ai_df, on="record_id", how="inner")
    .withColumn(
        "prediction_match",
        F.col("baseline_sentiment") == F.col("ai_sentiment")
    )
)

# Save the final Gold table
(
    final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_ai_feedback_enriched")
)

print("Final AI-enriched Gold table created.")

spark.table("gold_ai_feedback_enriched").select(
    "record_id",
    "product_category",
    "rating",
    "baseline_sentiment",
    "ai_sentiment",
    "sentiment_confidence",
    "feedback_theme",
    "prediction_match"
).orderBy("record_id").show(20, truncate=False)


In [ ]:
from pyspark.sql import functions as F

gold_df = spark.table("gold_ai_feedback_enriched")

print("Total rows:", gold_df.count())
print("Total columns:", len(gold_df.columns))
print("Columns:", gold_df.columns)

gold_df.groupBy("ai_sentiment").agg(
    F.count("*").alias("review_count"),
    F.round(
        F.avg("rating"), 2
    ).alias("average_rating")
).orderBy("ai_sentiment").show()
